# Certainty Evidence Tree: End-to-End Demo

This notebook demonstrates the full certainty delivery chain:

1. Define entities and rules with `condition_weights`
2. Register rules via `FileAuthoringRegistry`
3. Evaluate via runtime service → auto certainty routing
4. Inspect: `certainty_summary` with ranked conditions and bottleneck
5. Inspect: narrative with `[bottleneck]` markers and `certainty_bottleneck`
6. Inspect: NL with weakest-condition sentence
7. Export → audit round-trip

## 0. Imports

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path
from pprint import pprint

from factpy_kernel.sdk import (
    SDKStore,
    Entity,
    Identity,
    Field,
    Rule,
    Pred,
    vars as sdk_vars,
)
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.audit import AuditQuery, load_audit_package
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session,
    close_runtime_session,
    reset_runtime_sessions_for_tests,
    write_runtime_fact,
    evaluate_runtime_derivation,
    explain_runtime_summary,
    explain_runtime_narrative,
    explain_runtime_nl,
    export_runtime_package,
)

## 1. Schema & Rule Definition

A simple `User` entity with a `tag` field. The child rule `q.vip_rule`
declares `condition_weights` — this is what triggers certainty routing.

In [ ]:
class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")

sdk = SDKStore([User])

# Define a rule with condition_weights.
# Weights express relative importance of each condition in the rule body.
with sdk_vars("u", "tag") as (u, tag):
    child_rule = Rule(
        id="q.vip_rule",
        version="1.0.0",
        select=[u, tag],
        where=[Pred("user:tag", u, tag)],
        expose=True,
        condition_weights={"b0.a0": 0.8},
    )

print("Rule:", child_rule.id)
print("Condition weights:", child_rule.condition_weights)

## 2. Registry Setup & Seed Data

Register the rule in a temporary registry, then seed a user with a fact.

In [ ]:
registry_dir = tempfile.mkdtemp(prefix="certainty_demo_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
registry.register_rule_spec(sdk._compile_rule_input(child_rule))

print("Registry root:", registry_dir)
print("Rule spec stored:", registry.read_rule_spec("q.vip_rule", "1.0.0") is not None)

In [ ]:
# Seed a user entity and a tag fact.
with sdk.batch() as tx:
    u1 = tx.entity(User, user_id="u-001", locale="en")
    u1.name.set("Alice")
    u1.tag.add("vip")
    tx.commit()

u1_ref = sdk.ref(User, user_id="u-001", locale="en")
print("User ref:", u1_ref)

## 3. Runtime Evaluate with Certainty Routing

Open a runtime session **with `registry_root`**. When a native derivation
references a child rule that has `condition_weights`, the
`CertaintyConfidenceKindResolver` automatically marks the candidate as
`confidence_kind="certainty"` at creation time.

In [ ]:
reset_runtime_sessions_for_tests()
session_resp = open_runtime_session({"registry_root": registry_dir})
session_id = session_resp["session"]["session_id"]

# Write the same fact into the runtime session's store.
write_runtime_fact(
    session_id,
    {"pred_id": "user:tag", "e_ref": u1_ref, "rest_terms": [["string", "vip"]]},
    kind="add",
)

# Evaluate a derivation that references the child rule via ruleref.
eval_resp = evaluate_runtime_derivation(
    session_id,
    {
        "derivation": {
            "derivation_id": "drv.certainty_demo",
            "version": "1.0.0",
            "target": "user:tag",
            "head_vars": ["$u", "$tag"],
            "where": [
                ["ruleref", "q.vip_rule", "1.0.0", ["$u", "$tag"]],
                ["eq", "$tag", "vip"],
            ],
            "mode": "native",
        }
    },
)

candidate = eval_resp["evaluation"]["candidates"][0]
candidate_id = candidate["candidate_id"]

print("Candidate ID:", candidate_id)
print("confidence_kind:", candidate["confidence_kind"])
print()
print(">>> confidence_kind is automatically set to 'certainty' by the resolver!")

## 4. Explain Summary: `certainty_summary`

The explain-summary endpoint returns the structured certainty breakdown:
per-condition `weight`, `impact` (weight × confidence), and the
`aggregate_certainty` (bottleneck = minimum weighted impact).

In [ ]:
summary_resp = explain_runtime_summary(
    session_id,
    {"kind": "candidate", "id": candidate_id},
)

print("=== Certainty Summary ===")
pprint(summary_resp["certainty_summary"])

## 5. Explain Narrative: Ranked `certainty_lines` + `certainty_bottleneck`

The narrative sorts conditions by impact (ascending) and marks the
bottleneck condition with `[bottleneck]`. It also produces a
machine-readable `certainty_bottleneck` key for downstream consumers.

In [ ]:
narrative_resp = explain_runtime_narrative(
    session_id,
    {"kind": "candidate", "id": candidate_id},
)
narrative = narrative_resp["narrative"]

print("=== Certainty Lines (ranked by impact) ===")
for line in narrative.get("certainty_lines", []):
    print(" ", line)

print()
print("=== Certainty Bottleneck (machine-readable) ===")
pprint(narrative.get("certainty_bottleneck"))

## 6. Explain NL: Weakest-Condition Sentence

The NL layer appends a 5th paragraph with a human-readable summary
of the weakest condition (bottleneck).

In [ ]:
nl_resp = explain_runtime_nl(
    session_id,
    {"kind": "candidate", "id": candidate_id},
)

print("=== NL Explain (all paragraphs) ===")
for i, para in enumerate(nl_resp["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {para}")
    print()

## 7. Audit Round-Trip

Accept the candidate, export an audit package, and verify the
certainty summary survives the round-trip.

In [ ]:
from factpy_kernel.service.runtime_v1 import accept_runtime_derivation

accept_resp = accept_runtime_derivation(
    session_id,
    {"candidate": candidate, "options": {"approved_by": "demo"}},
)
print("Accept:", accept_resp["ok"])

with tempfile.TemporaryDirectory() as pkg_dir:
    export_resp = export_runtime_package(
        session_id,
        {"out_dir": pkg_dir, "package_kind": "audit"},
    )
    print("Export:", export_resp["ok"])

    # Load and query the audit package.
    package = load_audit_package(pkg_dir)
    audit_query = AuditQuery(package)

    audit_cs = audit_query.get_candidate_certainty_summary(candidate_id)
    print()
    print("=== Audit Certainty Summary ===")
    pprint(audit_cs)

    audit_narrative = audit_query.get_candidate_evidence_tree_narrative(candidate_id)
    print()
    print("=== Audit Certainty Lines ===")
    if audit_narrative:
        for line in audit_narrative.get("certainty_lines", []):
            print(" ", line)

    # Verify parity: runtime and audit certainty should match.
    print()
    runtime_cs = summary_resp["certainty_summary"]
    print("Runtime == Audit:", audit_cs == runtime_cs)

## 8. Negative Case: No `condition_weights` → `confidence_kind="none"`

When a rule does **not** declare `condition_weights`, the resolver
falls back to `"none"` and no certainty summary is produced.

In [ ]:
# Register a rule WITHOUT condition_weights.
with sdk_vars("u", "tag") as (u, tag):
    plain_rule = Rule(
        id="q.plain_rule",
        version="1.0.0",
        select=[u, tag],
        where=[Pred("user:tag", u, tag)],
        expose=True,
        # No condition_weights!
    )
registry.register_rule_spec(sdk._compile_rule_input(plain_rule))

eval_resp2 = evaluate_runtime_derivation(
    session_id,
    {
        "derivation": {
            "derivation_id": "drv.plain_demo",
            "version": "1.0.0",
            "target": "user:tag",
            "head_vars": ["$u", "$tag"],
            "where": [
                ["ruleref", "q.plain_rule", "1.0.0", ["$u", "$tag"]],
                ["eq", "$tag", "vip"],
            ],
            "mode": "native",
        }
    },
)

plain_candidate = eval_resp2["evaluation"]["candidates"][0]
print("Plain candidate confidence_kind:", plain_candidate["confidence_kind"])

plain_summary = explain_runtime_summary(
    session_id,
    {"kind": "candidate", "id": plain_candidate["candidate_id"]},
)
print("Certainty summary:", plain_summary.get("certainty_summary"))
print()
print(">>> No condition_weights => no certainty routing => certainty_summary is null.")

In [ ]:
# Cleanup
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
print("Session closed.")

## Summary

| Step | What happens |
|------|--------------|
| Rule declaration | `condition_weights={"b0.a0": 0.8}` on child rule |
| Registry | Rule payload stored with weights |
| Evaluate | `CertaintyConfidenceKindResolver` checks eligibility → `confidence_kind="certainty"` |
| Explain summary | `certainty_summary` with per-condition `impact` and `aggregate_certainty` |
| Explain narrative | `certainty_lines` sorted by impact, `[bottleneck]` marked |
| Explain NL | 5th paragraph with weakest-condition sentence |
| Audit export | `certainty_summaries.jsonl` materialized at export time |
| Audit query | `AuditQuery.get_candidate_certainty_summary()` reads materialized value |
| No weights | Falls back to `confidence_kind="none"`, no certainty delivery |